<a href="https://colab.research.google.com/github/leminhohoho/context-bert4rec/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
%pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.6 MB/s eta 0:00:00


In [5]:
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [6]:
dev = True

In [7]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()

        self.pe = nn.Embedding(max_len, d_model) # T, C

    def forward(self, x):
        batch_size = x.size(0)
        return self.pe.weight.unsqueeze(0).repeat(batch_size, 1, 1) # T, C -> B, T, C

class BERT4RecEmbedding(nn.Module):
    def __init__(self, embed_size, max_len, dropout=0.1):
        super().__init__()

        self.pe = PositionalEmbedding(max_len, embed_size)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        pos = self.pe(x) # B, T
        return self.dropout(x + pos)

if __name__ == "__main__" and dev:
    max_len = 5
    d_model = 4
    batch_size = 2

    model = BERT4RecEmbedding(d_model,max_len)
    x = torch.randn(batch_size, max_len, d_model)

    out = model(x)

    print(x.shape)
    print(x)
    print(out.shape)
    print(out)

    # NOTE: Padding example
    x = torch.randn(batch_size, max_len-2, d_model)
    x = torch.nn.functional.pad(x, (0 , 0, 0, 2))
    padding_mask = (x.abs().sum(dim=-1) == 0)

    print(x.shape)
    print(x)
    print(padding_mask.shape)
    print(padding_mask)


torch.Size([2, 5, 4])
tensor([[[ 7.6850e-01, -9.0607e-02, -2.8431e-01, -7.0465e-02],
         [ 4.8588e-01, -1.2525e+00, -7.0583e-01,  1.5333e+00],
         [ 1.0144e-01, -9.3756e-01, -7.3390e-01,  1.8173e+00],
         [-5.4324e-01, -1.2426e+00, -1.5990e+00, -1.0682e+00],
         [ 1.7796e+00, -4.5253e-01,  4.5979e-01,  9.6293e-01]],

        [[ 6.1216e-01, -5.0877e-01,  5.5737e-01, -2.1771e-02],
         [ 2.2661e-01, -6.4376e-01,  2.1233e+00,  1.2933e+00],
         [-1.8292e-02, -1.3949e+00, -9.6624e-01,  5.7522e-01],
         [-1.8593e-01,  1.0787e+00, -2.9919e-01, -1.5457e+00],
         [-1.0478e+00, -3.0741e-04,  1.8563e+00, -8.7650e-01]]])
torch.Size([2, 5, 4])
tensor([[[-0.5483, -0.3626, -0.0000,  1.9794],
         [-0.7946, -1.6566, -2.0332,  1.7186],
         [-0.1461, -2.3424,  0.1033,  1.0374],
         [-0.0380, -0.4398, -0.4906,  0.0000],
         [ 0.0000, -0.3350, -0.3710,  0.1930]],

        [[-0.7220, -0.8273,  0.1854,  2.0335],
         [-1.0827, -0.9803,  1.1103,  

In [8]:
class BERT(nn.Module):
    def __init__(self, max_len, n_layers, n_heads, hidden, dropout=0.1):
        super().__init__()
        self.embedding = BERT4RecEmbedding(
            embed_size=hidden,
            max_len=max_len,
            dropout=dropout,
        )

        self.mask_token = nn.Parameter(torch.randn(hidden))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=n_heads,
            batch_first=True,
            dropout=dropout,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=n_layers,
        )

        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, padding_mask=None, masked_positions=None):
        if padding_mask is None:
            padding_mask = (x.abs().sum(dim=-1) == 0)

        x = self.embedding(x)

        if masked_positions is not None:
            x = torch.where(
                masked_positions.unsqueeze(-1),
                self.mask_token.view(1, 1, -1),
                x,
            )

        x = self.encoder(x, src_key_padding_mask=padding_mask)
        x = self.dropout(x)

        return x

if __name__ == "__main__" and dev:
    batch_size = 2
    max_len = 4
    hidden = 32
    n_layers = 3
    n_heads = 4

    model = BERT(max_len=max_len, hidden=hidden, n_layers=n_layers, n_heads=n_heads)

    x = torch.randn(batch_size, max_len, hidden)

    out = model(x)

    print(out.shape)
    print(out)

torch.Size([2, 4, 32])
tensor([[[ 2.1140e-01, -3.6499e-01,  1.0782e+00,  1.6522e+00,  2.5207e+00,
          -2.2357e-01, -1.6740e+00, -0.0000e+00, -4.0678e-01, -7.6662e-01,
           2.8265e+00, -3.2725e-01, -5.2912e-01, -1.7804e-01,  3.8273e-01,
          -6.9835e-01,  9.0120e-01, -1.5293e+00,  5.8970e-01, -1.9744e+00,
          -1.5899e+00,  6.0379e-01,  9.2508e-01,  5.8451e-01, -9.8023e-01,
          -4.7328e-01,  4.0249e-01, -1.1983e+00,  1.0366e+00,  3.2972e-01,
          -7.3289e-01,  2.2836e-01],
         [ 1.7744e-01, -1.1110e+00,  8.5553e-01,  0.0000e+00, -9.1469e-01,
          -2.0699e-01, -1.2609e+00, -1.1498e+00,  0.0000e+00,  3.1187e-01,
           1.1274e+00, -7.4359e-01,  3.1225e-01, -1.5551e+00,  6.7220e-01,
          -1.6456e-01,  5.3926e-01,  5.1186e-01,  6.4401e-02, -0.0000e+00,
          -1.2437e+00, -2.4479e-01,  5.0362e-01, -3.7767e-02, -1.2716e+00,
          -6.5729e-01, -9.0793e-01,  1.1952e+00,  2.4382e-01,  9.4710e-01,
           3.4810e+00, -7.6980e-01],
   

In [9]:
import torch.nn.functional as F

class UserEncoder(nn.Module):
    def __init__(self, max_len, n_layers, n_heads, hidden, embed_size, dropout=0.1):
        super().__init__()

        self.encoder = BERT(
            max_len=max_len,
            n_layers=n_layers,
            n_heads=n_heads,
            hidden=hidden,
            dropout=dropout,
        )

        self.proj = nn.Linear(hidden, embed_size)

    def forward(self, x):
        x = self.encoder(x)
        x = self.proj(x[:, -1, :])

        return x

class CandidateGenerator(nn.Module):
    def __init__(self, max_len, n_layers, n_heads, hidden, dropout=0.1):
        super().__init__()

        self.user_encoder = UserEncoder(
            max_len=max_len,
            n_layers=n_layers,
            n_heads=n_heads,
            hidden=hidden,
            dropout=dropout,
        )

    def forward(self, user_seq, targets):
        print(user_seq.shape)
        user_embedding = self.user_encoder(user_seq)
        user_embedding = user_embedding.unsqueeze(1)
        user_embedding = F.normalize(user_embedding, dim=-1)
        targets = F.normalize(targets, dim=-1)
        targets = targets.transpose(1,2)

        print(user_embedding.shape)
        print(targets.shape)

        logits = torch.bmm(user_embedding, targets)

        return logits

if __name__ == "__main__" and dev:
    batch_size = 2
    max_len = 4
    hidden = 16
    n_layers = 3
    n_heads = 4
    items_length = 100

    model = CandidateGenerator(max_len=max_len, hidden=hidden, n_layers=n_layers, n_heads=n_heads)

    x = torch.randn(batch_size, max_len, hidden)
    targets = torch.randn(batch_size, items_length, hidden)

    out = model(x, targets)

    print(out.shape)
    print(out)

torch.Size([2, 4, 16])
torch.Size([2, 1, 16])
torch.Size([2, 16, 100])
torch.Size([2, 1, 100])
tensor([[[ 0.0025, -0.0340,  0.1043, -0.2661,  0.0421,  0.1434, -0.0201,
           0.1633, -0.0321, -0.3246,  0.3650, -0.1803, -0.1048, -0.4982,
          -0.0044,  0.0068, -0.1740, -0.0690, -0.0476,  0.3314, -0.0249,
          -0.1769,  0.2101, -0.0576, -0.1474,  0.0534,  0.0974, -0.0877,
           0.2894,  0.1349, -0.2745, -0.2902, -0.0679, -0.3965,  0.1495,
          -0.0496, -0.3673,  0.2750, -0.0663,  0.4280,  0.0169,  0.2403,
           0.2643, -0.2946,  0.1348,  0.1816,  0.4030,  0.0597,  0.0853,
          -0.0632, -0.0855, -0.5059,  0.2237, -0.1929,  0.2244,  0.2089,
          -0.3356,  0.3528, -0.1758,  0.2519,  0.0665,  0.3820, -0.1447,
          -0.1219, -0.1025,  0.1452,  0.0562,  0.0245, -0.3840, -0.5352,
          -0.0219,  0.0315,  0.2730, -0.1680, -0.1583, -0.3117,  0.0258,
           0.2590, -0.0581, -0.2819,  0.0293, -0.4362,  0.2967, -0.3715,
          -0.2290,  0.4853,  